In [0]:
# Databricks notebook source
# Aggregates data quality expectation results into an audit table.

from pyspark.sql.functions import (
    col, explode, current_timestamp, lit, sum as ssum, max as smax,
)

CATALOG = "ecom_dev"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.ops")

# The event_log() TVF needs a table owned by the pipeline you're inspecting.
PIPELINE_PROBES = {
    "silver": f"{CATALOG}.silver.sv_orders",
    "gold":   f"{CATALOG}.gold.fct_order_items",
}

frames = []

for layer, probe_table in PIPELINE_PROBES.items():
    try:
        df = spark.sql(f"""
            SELECT
              timestamp,
              details:flow_progress.data_quality.expectations AS expectations
            FROM event_log(TABLE({probe_table}))
            WHERE event_type = 'flow_progress'
              AND details:flow_progress.data_quality.expectations IS NOT NULL
        """)

        parsed = (
            df.selectExpr(
                "timestamp",
                "from_json(expectations, 'array<struct<name:string,dataset:string,"
                "passed_records:bigint,failed_records:bigint>>') AS exp",
            )
            .select("timestamp", explode("exp").alias("e"))
            .select(
                lit(layer).alias("layer"),
                col("e.dataset").alias("dataset"),
                col("e.name").alias("expectation"),
                col("e.passed_records").alias("passed"),
                col("e.failed_records").alias("failed"),
                col("timestamp"),
            )
        )
        frames.append(parsed)
        print(f"{layer}: event log read OK")
    except Exception as e:
        print(f"{layer}: could not read event log — {type(e).__name__}: {e}")

if not frames:
    raise Exception("No pipeline event logs could be read")

combined = frames[0]
for f in frames[1:]:
    combined = combined.unionByName(f)

# Keep only the latest observation per expectation.
latest = (
    combined.groupBy("layer", "dataset", "expectation")
    .agg(
        smax("timestamp").alias("last_seen"),
        ssum("passed").alias("total_passed"),
        ssum("failed").alias("total_failed"),
    )
    .withColumn("report_run_at", current_timestamp())
)

(latest.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.ops.dq_expectation_report"))

print(f"\nWrote {latest.count()} expectation records to {CATALOG}.ops.dq_expectation_report")
display(latest.orderBy(col("total_failed").desc()))